In [1]:
# ============================================================
# SECTION 3 EXPERIMENTS — FINAL PAPER EVALUATION
# ============================================================

import os, json, pickle, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"
OUT = "/kaggle/working/"

os.makedirs(OUT, exist_ok=True)

print("Section 3 experiments started")


# ============================================================
# LOAD CONFIG
# ============================================================

with open(f"{BASE}/cps-config/config.json") as f:
    CFG = json.load(f)

TRAIN_IDS = CFG["engine_partition"]["train"]
VAL_IDS = CFG["engine_partition"]["validation"]
TEST_IDS = CFG["engine_partition"]["test"]


# ============================================================
# LOAD FEATURE DATA
# ============================================================

df = pd.read_csv(
    f"{BASE}/v2-phase3-features/v2_phase3_feature_dataset.csv"
)

with open(f"{BASE}/v2-phase3-features/v2_feature_cols.json") as f:
    feature_cols = json.load(f)

scaler = pickle.load(
    open(f"{BASE}/v2-phase3-features/v2_scaler.pkl","rb")
)

print("Feature dataset loaded:", df.shape)


# ============================================================
# LOAD MODELS
# ============================================================

best_model = pickle.load(
    open(f"{BASE}/v2-baseline-results/v2_best_centralized.pkl","rb")
)

weights_df = pd.read_csv(
    f"{BASE}/v2-node-predictions/v2_node_f1_weights.csv"
)

node_weights = dict(zip(weights_df.node_id, weights_df.weight))

node_models = {}

for nid in node_weights:
    path = f"{BASE}/v2-node-predictions/v2_{nid}_xgb.pkl"
    node_models[nid] = pickle.load(open(path,"rb"))

print("Models loaded")


# ============================================================
# LABEL COLUMN
# ============================================================

top_models = pd.read_csv(
    f"{BASE}/v2-baseline-results/v2_top_models.csv"
)

best_window = int(top_models.iloc[0]["window"])
LABEL_COL = f"label_w{best_window}"

print("Using label:", LABEL_COL)


# ============================================================
# DISTRIBUTED PREDICTION
# ============================================================

def distributed_predict(X):

    votes = []

    for nid, model in node_models.items():

        probs = model.predict_proba(X)[:,1]
        flags = (probs > 0.5).astype(int)
        weight = node_weights[nid]

        votes.append(flags * weight)

    weighted = sum(votes)

    return (weighted > 0.5).astype(int)


# ============================================================
# A. TEST PARTITION EVALUATION
# ============================================================

print("\nRunning TEST partition experiment")

test_df = df[df.engine_id.isin(TEST_IDS)].copy()

X_test = scaler.transform(test_df[feature_cols])
y_test = test_df[LABEL_COL].values

cent_pred = best_model.predict(X_test)
dist_pred = distributed_predict(X_test)

test_results = pd.DataFrame({

    "system":["centralized","distributed"],
    "precision":[
        precision_score(y_test,cent_pred),
        precision_score(y_test,dist_pred)
    ],
    "recall":[
        recall_score(y_test,cent_pred),
        recall_score(y_test,dist_pred)
    ],
    "f1":[
        f1_score(y_test,cent_pred),
        f1_score(y_test,dist_pred)
    ]

})

test_results.to_csv(f"{OUT}/test_system_comparison.csv",index=False)

print("Test evaluation complete")
print(test_results)


# ============================================================
# B. WILCOXON SIGNED-RANK TEST
# ============================================================

print("\nRunning Wilcoxon test")

engine_scores = []

for eid in TEST_IDS:

    e_df = test_df[test_df.engine_id == eid]

    X = scaler.transform(e_df[feature_cols])
    y = e_df[LABEL_COL].values

    c = f1_score(y, best_model.predict(X))
    d = f1_score(y, distributed_predict(X))

    engine_scores.append({
        "engine_id":eid,
        "centralized_f1":c,
        "distributed_f1":d
    })

engine_scores = pd.DataFrame(engine_scores)

stat,p = wilcoxon(
    engine_scores["centralized_f1"],
    engine_scores["distributed_f1"]
)

wilcoxon_df = pd.DataFrame({

    "statistic":[stat],
    "p_value":[p]

})

wilcoxon_df.to_csv(f"{OUT}/wilcoxon_test_results.csv",index=False)

print("Wilcoxon p-value:",p)


# ============================================================
# C. DETECTION DELAY
# ============================================================

print("\nComputing detection delay")

delay_rows = []

for eid in TEST_IDS:

    e_df = test_df[test_df.engine_id == eid]

    X = scaler.transform(e_df[feature_cols])
    preds = distributed_predict(X)

    fault_idx = np.where(e_df[LABEL_COL].values==1)[0]

    if len(fault_idx)==0:
        continue

    fault_start = fault_idx[0]

    pred_idx = np.where(preds==1)[0]

    if len(pred_idx)==0:
        continue

    first_pred = pred_idx[0]

    delay = first_pred - fault_start

    delay_rows.append({
        "engine_id":eid,
        "detection_delay":delay
    })

delay_df = pd.DataFrame(delay_rows)

delay_df.to_csv(f"{OUT}/detection_delay_results.csv",index=False)

print("Detection delay mean:",delay_df.detection_delay.mean())


# ============================================================
# D. FD003 GENERALIZATION TEST
# ============================================================

# ============================================================
# D. FD003 GENERALIZATION TEST (FIXED)
# ============================================================

print("\nRunning FD003 generalization test")

fd003 = pd.read_csv(
    f"{BASE}/cmapss-turbofan/train_FD003.txt",
    delim_whitespace=True,
    header=None
)

fd003.columns = ["engine_id","cycle","op1","op2","op3"] + \
                [f"s{i}" for i in range(1,22)]

# remove operating settings
fd003 = fd003.drop(columns=["op1","op2","op3"])

# keep only active sensors used in training
active_sensors = [c for c in feature_cols if "_" not in c]

fd003 = fd003[["engine_id","cycle"] + active_sensors]

# compute max cycle
fd003["max_cycle"] = fd003.groupby("engine_id")["cycle"].transform("max")

# create label
fd003["label"] = (
    fd003["cycle"] >= fd003["max_cycle"] - best_window
).astype(int)

# ------------------------------------------------
# RECREATE FEATURE ENGINEERING (same as Notebook 3)
# ------------------------------------------------

for s in active_sensors:

    fd003[f"{s}_mean5"] = (
        fd003.groupby("engine_id")[s]
        .rolling(5,min_periods=1)
        .mean()
        .reset_index(level=0,drop=True)
    )

    fd003[f"{s}_std5"] = (
        fd003.groupby("engine_id")[s]
        .rolling(5,min_periods=1)
        .std()
        .reset_index(level=0,drop=True)
    ).fillna(0)

    fd003[f"{s}_slope5"] = (
        fd003.groupby("engine_id")[s]
        .diff()
    ).fillna(0)

# fill any remaining NaN
fd003 = fd003.fillna(0)

# prediction
X_fd003 = scaler.transform(fd003[feature_cols])
y_fd003 = fd003["label"].values

fd003_pred = distributed_predict(X_fd003)

fd003_results = pd.DataFrame({

    "precision":[precision_score(y_fd003,fd003_pred)],
    "recall":[recall_score(y_fd003,fd003_pred)],
    "f1":[f1_score(y_fd003,fd003_pred)]

})

fd003_results.to_csv(
    f"{OUT}/fd003_generalization_results.csv",
    index=False
)

print("FD003 results")
print(fd003_results)

# ============================================================
# FINAL OUTPUT CHECK
# ============================================================

print("\nGenerated files:")

print(os.listdir(OUT))

Section 3 experiments started
Feature dataset loaded: (20631, 65)
Models loaded
Using label: label_w20

Running TEST partition experiment
Test evaluation complete
        system  precision    recall        f1
0  centralized   0.870748  0.812698  0.840722
1  distributed   0.798851  0.882540  0.838612

Running Wilcoxon test
Wilcoxon p-value: 0.52447509765625

Computing detection delay
Detection delay mean: -3.2

Running FD003 generalization test
FD003 results
   precision    recall        f1
0   0.873585  0.661429  0.752846

Generated files:
['fd003_generalization_results.csv', 'test_system_comparison.csv', '__notebook__.ipynb', 'wilcoxon_test_results.csv', 'detection_delay_results.csv']
